In [1]:
# ============================================
# RF + SMOTE (adaptif k) for 4 targets
# LOSO vs Stratified 5-Fold vs Stratified 10-Fold
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import statsmodels.stats.proportion as smp

# --- 1) Veri yükle ---
file_path = "ML_Analysis_V3.xlsx"  # kendi dosya yolunu yaz
df = pd.read_excel(file_path, sheet_name="Sheet1")

targets = ["Cervical Lordosis Risk","Kyphosis Risk","Lumbar Lordosis Risk","Scoliosis Risk"]
feature_cols = [c for c in df.columns if c not in targets]

# --- 2) Yardımcı fonksiyon: tek fold eğitimi ---
def fit_predict_one_fold(X_train, y_train, X_test, rf_params):
    # minority sınıf sayısı
    cls_counts = y_train.value_counts().to_dict()
    if 0 not in cls_counts: cls_counts[0] = 0
    if 1 not in cls_counts: cls_counts[1] = 0
    minority_n = min(cls_counts[0], cls_counts[1])
    # SMOTE için k_neighbors seçimi
    k = max(1, min(5, minority_n - 1))  # 1 ile 5 arasında
    pipe = Pipeline([
        ('smote', SMOTE(k_neighbors=k, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    return pipe.predict(X_test)

# --- 3) CV şeması ile değerlendirme ---
def evaluate_with_scheme(X, y, splitter, scheme_name):
    y_true, y_pred = [], []
    if isinstance(splitter, StratifiedKFold):
        splits = splitter.split(X, y)
    else:
        splits = splitter.split(X)
    for tr, te in splits:
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        pred = fit_predict_one_fold(X_tr, y_tr, X_te, dict(n_estimators=100, random_state=42))
        y_true.extend(y_te)
        y_pred.extend(pred)
    # metrikler
    acc = accuracy_score(y_true, y_pred)
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    n = len(y_true)
    k = int((np.array(y_true) == np.array(y_pred)).sum())
    ci_lo, ci_hi = smp.proportion_confint(k, n, alpha=0.05, method='wilson')
    row = {
        "Scheme": scheme_name,
        "Accuracy": acc,
        "95% CI Low": ci_lo,
        "95% CI High": ci_hi,
        "Weighted Precision": rep["weighted avg"]["precision"],
        "Weighted Recall": rep["weighted avg"]["recall"],
        "Weighted F1": rep["weighted avg"]["f1-score"],
        "Support": int(cm.sum()),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }
    return row, cm

# --- 4) Tüm hedefler için çalıştır ---
results = []
cms = {}
for target in targets:
    X = df[feature_cols].copy()
    y = df[target].copy()

    # CV şemaları
    cv_loso = LeaveOneOut()
    cv_s5   = StratifiedKFold(n_splits=5,  shuffle=True, random_state=42)
    cv_s10  = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # 1) LOSO
    row_loso, cm_loso = evaluate_with_scheme(X, y, cv_loso, f"{target} – LOSO")
    results.append({"Target": target, **row_loso})
    cms[f"{target} – LOSO"] = cm_loso

    # 2) Stratified 5-Fold
    row_s5, cm_s5 = evaluate_with_scheme(X, y, cv_s5, f"{target} – Stratified 5-Fold")
    results.append({"Target": target, **row_s5})
    cms[f"{target} – Stratified 5-Fold"] = cm_s5

    # 3) Stratified 10-Fold
    try:
        row_s10, cm_s10 = evaluate_with_scheme(X, y, cv_s10, f"{target} – Stratified 10-Fold")
        results.append({"Target": target, **row_s10})
        cms[f"{target} – Stratified 10-Fold"] = cm_s10
    except Exception as e:
        print(f"{target}: Stratified 10-Fold uygulanamadı ({e})")

# --- 5) Master tablo ---
summary_master = pd.DataFrame(results)
print("=== RF + SMOTE (LOSO vs 5-Fold vs 10-Fold) – Master Tablo ===")
print(summary_master)

# CSV kaydetmek için:
# summary_master.to_csv("RF_LOSO_5F_10F_AllTargets.csv", index=False)

# Confusion matrix'ler cms dict'inde tutuluyor, örn:
# print(cms["Cervical Lordosis Risk – LOSO"])

C:\Anaconda\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
C:\Anaconda\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Anaconda\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=10.
  warnings.warn(


=== RF + SMOTE (LOSO vs 5-Fold vs 10-Fold) – Master Tablo ===
                    Target                                       Scheme  \
0   Cervical Lordosis Risk                Cervical Lordosis Risk – LOSO   
1   Cervical Lordosis Risk   Cervical Lordosis Risk – Stratified 5-Fold   
2   Cervical Lordosis Risk  Cervical Lordosis Risk – Stratified 10-Fold   
3            Kyphosis Risk                         Kyphosis Risk – LOSO   
4            Kyphosis Risk            Kyphosis Risk – Stratified 5-Fold   
5            Kyphosis Risk           Kyphosis Risk – Stratified 10-Fold   
6     Lumbar Lordosis Risk                  Lumbar Lordosis Risk – LOSO   
7     Lumbar Lordosis Risk     Lumbar Lordosis Risk – Stratified 5-Fold   
8     Lumbar Lordosis Risk    Lumbar Lordosis Risk – Stratified 10-Fold   
9           Scoliosis Risk                        Scoliosis Risk – LOSO   
10          Scoliosis Risk           Scoliosis Risk – Stratified 5-Fold   
11          Scoliosis Risk          Sc

In [2]:
import os, math, re
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

os.makedirs("supp_figs", exist_ok=True)
os.makedirs("supp_csv", exist_ok=True)

def slug(s):
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "-", s)
    return re.sub(r"-+", "-", s).strip("-")

def plot_fold_confusions(confusions, title, save_basename, cmap="Blues"):
    """
    confusions: list of 2x2 numpy arrays (fold order)
    title: figure title
    save_basename: without extension; files written to supp_figs/
    """
    n = len(confusions)
    if n == 0:
        return
    # 5 sütunlu grid genelde güzel; satır sayısını otomatik ayarla
    cols = 5 if n >= 5 else n
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.4, rows*2.4))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])

    for i in range(rows*cols):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        if i < n:
            cm = confusions[i]
            im = ax.imshow(cm, interpolation="nearest", cmap=cmap, vmin=0)
            # anotasyon
            for (yy, xx), val in np.ndenumerate(cm):
                ax.text(xx, yy, str(int(val)), ha="center", va="center", fontsize=8)
            ax.set_xticks([0,1]); ax.set_yticks([0,1])
            ax.set_xticklabels(["Pred 0","Pred 1"], fontsize=8, rotation=0)
            ax.set_yticklabels(["True 0","True 1"], fontsize=8)
            ax.set_title(f"Fold {i+1}", fontsize=8)
        else:
            ax.axis("off")

    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.subplots_adjust(top=0.90)

    png_path = os.path.join("supp_figs", f"{save_basename}.png")
    svg_path = os.path.join("supp_figs", f"{save_basename}.svg")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.close(fig)

def save_fold_level_preds(y_true_folds, y_pred_folds, save_basename):
    """
    Her fold için y_true/y_pred’i tek CSV’de uzun formatta saklar.
    """
    import pandas as pd
    rows = []
    for i, (yt, yp) in enumerate(zip(y_true_folds, y_pred_folds), start=1):
        for t, p in zip(list(yt), list(yp)):
            rows.append({"fold": i, "y_true": int(t), "y_pred": int(p)})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join("supp_csv", f"{save_basename}.csv"), index=False)

In [3]:
def evaluate_with_scheme_foldwise(X, y, splitter, scheme_name):
    """
    Döndürür:
      - row (genel özet metrikler, Wilson CI ile)
      - cm_total (toplam confusion)
      - fold_confusions (liste: her fold için 2x2 cm)
      - y_true_folds / y_pred_folds (liste)
    """
    if isinstance(splitter, StratifiedKFold):
        splits = splitter.split(X, y)
    else:
        splits = splitter.split(X)

    y_true_all, y_pred_all = [], []
    fold_confusions = []
    y_true_folds, y_pred_folds = [], []

    for tr, te in splits:
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        pred = fit_predict_one_fold(X_tr, y_tr, X_te, dict(n_estimators=100, random_state=42))

        # fold bazında kaydet
        y_true_folds.append(np.array(y_te))
        y_pred_folds.append(np.array(pred))
        cm_fold = confusion_matrix(y_te, pred, labels=[0,1])
        fold_confusions.append(cm_fold)

        # genel
        y_true_all.extend(y_te)
        y_pred_all.extend(pred)

    # genel metrikler
    acc = accuracy_score(y_true_all, y_pred_all)
    rep = classification_report(y_true_all, y_pred_all, output_dict=True, zero_division=0)
    cm_total = confusion_matrix(y_true_all, y_pred_all, labels=[0,1])
    tn, fp, fn, tp = cm_total.ravel()
    n = len(y_true_all)
    k = int((np.array(y_true_all) == np.array(y_pred_all)).sum())
    ci_lo, ci_hi = smp.proportion_confint(k, n, alpha=0.05, method='wilson')

    row = {
        "Scheme": scheme_name,
        "Accuracy": acc,
        "95% CI Low": ci_lo, "95% CI High": ci_hi,
        "Weighted Precision": rep["weighted avg"]["precision"],
        "Weighted Recall": rep["weighted avg"]["recall"],
        "Weighted F1": rep["weighted avg"]["f1-score"],
        "Support": int(cm_total.sum()),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }
    return row, cm_total, fold_confusions, y_true_folds, y_pred_folds

In [4]:
results = []
cms_total = {}  # toplam cm
for target in targets:
    X = df[feature_cols].copy()
    y = df[target].copy()

    cv_loso = LeaveOneOut()
    cv_s5   = StratifiedKFold(n_splits=5,  shuffle=True, random_state=42)
    cv_s10  = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # 1) LOSO
    row_loso, cm_loso, fold_cms_loso, yt_loso, yp_loso =
        evaluate_with_scheme_foldwise(X, y, cv_loso, "LOSO")
    results.append({"Target": target, **row_loso})
    cms_total[f"{target} – LOSO"] = cm_loso

    base = f"{slug(target)}__loso"
    plot_fold_confusions(fold_cms_loso,
                         title=f"{target} – LOSO: Fold-level confusion matrices",
                         save_basename=base)
    save_fold_level_preds(yt_loso, yp_loso, save_basename=base)

    # 2) Stratified 5-Fold
    row_s5, cm_s5, fold_cms_s5, yt_s5, yp_s5 =
        evaluate_with_scheme_foldwise(X, y, cv_s5, "Stratified 5-Fold")
    results.append({"Target": target, **row_s5})
    cms_total[f"{target} – Stratified 5-Fold"] = cm_s5

    base = f"{slug(target)}__s5"
    plot_fold_confusions(fold_cms_s5,
                         title=f"{target} – Stratified 5-Fold: Fold-level confusion matrices",
                         save_basename=base)
    save_fold_level_preds(yt_s5, yp_s5, save_basename=base)

    # 3) Stratified 10-Fold (küçük azınlıkta çalışmayabilir)
    try:
        row_s10, cm_s10, fold_cms_s10, yt_s10, yp_s10 =
            evaluate_with_scheme_foldwise(X, y, cv_s10, "Stratified 10-Fold")
        results.append({"Target": target, **row_s10})
        cms_total[f"{target} – Stratified 10-Fold"] = cm_s10

        base = f"{slug(target)}__s10"
        plot_fold_confusions(fold_cms_s10,
                             title=f"{target} – Stratified 10-Fold: Fold-level confusion matrices",
                             save_basename=base)
        save_fold_level_preds(yt_s10, yp_s10, save_basename=base)
    except Exception as e:
        print(f"{target}: Stratified 10-Fold uygulanamadı ({e})")


SyntaxError: invalid syntax (3011629161.py, line 12)

In [5]:
summary_master = pd.DataFrame(results)
print("=== RF + SMOTE (LOSO vs 5-Fold vs 10-Fold) – Master Tablo ===")
print(summary_master)
# summary_master.to_csv("RF_LOSO_5F_10F_AllTargets.csv", index=False)

=== RF + SMOTE (LOSO vs 5-Fold vs 10-Fold) – Master Tablo ===
                    Target                                       Scheme  \
0   Cervical Lordosis Risk                Cervical Lordosis Risk – LOSO   
1   Cervical Lordosis Risk   Cervical Lordosis Risk – Stratified 5-Fold   
2   Cervical Lordosis Risk  Cervical Lordosis Risk – Stratified 10-Fold   
3            Kyphosis Risk                         Kyphosis Risk – LOSO   
4            Kyphosis Risk            Kyphosis Risk – Stratified 5-Fold   
5            Kyphosis Risk           Kyphosis Risk – Stratified 10-Fold   
6     Lumbar Lordosis Risk                  Lumbar Lordosis Risk – LOSO   
7     Lumbar Lordosis Risk     Lumbar Lordosis Risk – Stratified 5-Fold   
8     Lumbar Lordosis Risk    Lumbar Lordosis Risk – Stratified 10-Fold   
9           Scoliosis Risk                        Scoliosis Risk – LOSO   
10          Scoliosis Risk           Scoliosis Risk – Stratified 5-Fold   
11          Scoliosis Risk          Sc